# <center> مشروع فردي. توقع تصنيف الدواء بناءً على المراجعة</center>



تحياتي، 
لقد تغلغل التعلم الآلي في جميع مجالات وتخصصات الدراسة تقريبًا. أحد المواضيع الساخنة هو استخدام معالجة اللغة الطبيعية وتحليل المشاعر لتحديد المعلومات الشخصية واستخراجها والاستفادة منها. توفر مجموعة بيانات UCI ML Drug Review مراجعات للمرضى حول أدوية محددة إلى جانب الحالات ذات الصلة ونظام تصنيف للمرضى من فئة 10 نجوم يعكس رضا المرضى بشكل عام. تم الحصول على البيانات عن طريق الزحف إلى مواقع مراجعة الأدوية عبر الإنترنت. 
تم نشر هذه البيانات في دراسة حول تحليل المشاعر لتجربة المخدرات على جوانب متعددة، على سبيل المثال. المشاعر المستفادة بشأن جوانب محددة مثل الفعالية والآثار الجانبية (راجع قسم الشكر والتقدير لمعرفة المزيد).
تم نشر مجموعة البيانات في الأصل على مستودع UCI Machine Learning: https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29
الاقتباس: 
فيليكس جراسر، سوريا كالومادي، هاجن مالبيرج، وسيباستيان زاونسيدر. 2018. تحليل المشاعر على أساس الجوانب لمراجعات الأدوية مع تطبيق التعلم عبر المجالات والبيانات. في وقائع المؤتمر الدولي لعام 2018 حول الصحة الرقمية (DH '18). إيه سي إم، نيويورك، نيويورك، الولايات المتحدة الأمريكية، 121-125.
يمكنك أيضًا تنزيله بسهولة من مجموعة بيانات kagle:
https://www.kaggle.com/jessicali9530/kuc-hackathon-winter-2018



لتبسيط تقييم المشروع سأتبع الخطة المقترحة:



#مخطط المشروع
1. شرح الميزة والبيانات
2. EDA وVDA والرؤى والتبعيات الموجودة
3. اختيار المقاييس 
4. المعالجة المسبقة للبيانات واختيار النموذج
5. التحقق من صحة وتعديل المعلمات الفائقة للنموذج
6. إنشاء ميزات جديدة ووصف هذه العملية
7. رسم منحنيات التدريب والتحقق من الصحة
8. التنبؤ بالعينات الاختبارية أو المحتجزة 
9. الاستنتاجات


# 1. شرح الميزات والبيانات



أولاً، قم بتحميل مجموعة البيانات 


In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

random_state = 42

PATH_TO_DATA = 'C:/Projects/Python/ODS_ml_course/indiv_proj/'
df_train = pd.read_csv(os.path.join(PATH_TO_DATA,
                                     'drugsComTrain_raw.csv'), parse_dates=["date"])
df_test = pd.read_csv(os.path.join(PATH_TO_DATA,
                                     'drugsComTest_raw.csv'), parse_dates=["date"])
df_train.drop('uniqueID', axis=1, inplace=True)
df_test.drop('uniqueID', axis=1, inplace=True)


دعونا نلقي نظرة على البيانات لدينا


In [ ]:
df_train.head()

In [ ]:
df_train.shape

In [ ]:
df_train.info()

In [ ]:
df_test.head()

In [ ]:
df_test.shape

In [ ]:
df_test.info()


الأعمدة المضمنة في مجموعة البيانات هذه هي:
1. اسم الدواء (الفئوي): اسم الدواء 
2. الشرط (الفئوي): اسم الشرط 
3. المراجعة (النص): مراجعة المريض 
4. التقييم (عددي): تقييم المريض 10 نجوم 
5. التاريخ (التاريخ): تاريخ إدخال المراجعة 
6. عدد مفيد (عددي): عدد المستخدمين الذين وجدوا المراجعة مفيدة
هيكل البيانات هو أن المريض يشتري دواءً يلبي حالته ويكتب مراجعة وتقييمًا للدواء الذي اشتراه. بعد ذلك، إذا قرأ الآخرون تلك المراجعة ووجدوها مفيدة، فسيقومون بالنقر فوق "عدد مفيد"، مما سيضيف 1 للمتغير.
يتم تقسيم البيانات إلى قسم قطار (75%) واختبار (25%).



وكانت المهام الأولية
1. التصنيف: هل يمكنك التنبؤ بحالة المريض بناءً على المراجعة؟
2. الانحدار: هل يمكنك التنبؤ بتصنيف الدواء بناءً على المراجعة؟
3. تحليل المشاعر: ما هي عناصر المراجعة التي تجعلها أكثر فائدة للآخرين؟ أي المرضى يميلون إلى الحصول على المزيد من المراجعات السلبية؟ هل يمكنك تحديد ما إذا كانت المراجعة إيجابية أم محايدة أم سلبية؟
4. تصور البيانات: ما هي أنواع الأدوية الموجودة؟ ما هي أنواع الحالات التي يعاني منها هؤلاء المرضى؟
متغير التصنيف ترتيبي. ومن المؤكد أنه سيكون هناك الكثير من الألم في $$. فليكن تصنيف المشاعر للمراجعات. 


In [ ]:
df_train['target'] = df_train['rating'].apply(lambda x: 0 if x < 5 else 1 if 4 < x < 8 else 2)
df_test['target'] = df_test['rating'].apply(lambda x: 0 if x < 5 else 1 if 4 < x < 8 else 2)

df_train.drop('rating', axis=1, inplace=True)
df_test.drop('rating', axis=1, inplace=True)


# 2. EDA وVDA والرؤى والتبعيات الموجودة



حان الوقت لتحميل العديد من المكتبات اللازمة لتحليل البيانات:


In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns


أولاً، كما لاحظت، هناك العديد من NaNs في عمود "الحالة" في مجموعتي البيانات.


In [ ]:
df_train[pd.isnull(df_train['condition'])].head()

In [ ]:
df_train[pd.isnull(df_train['condition'])].shape, df_test[pd.isnull(df_test['condition'])].shape


حسنا، المبلغ منخفض. ويبدو أن هذه NaNs تفتقد القيم. سنقوم بحذفها لاحقا.


بخير. نلقي نظرة على الميزات لدينا.


In [ ]:
df_train.describe(include=['object','bool'])

In [ ]:
df_test.describe(include=['object','bool'])


أما بالنسبة لعمود الشرط


In [ ]:
vc_condition_train = df_train['condition'].value_counts()
vc_condition_test = df_test['condition'].value_counts()

print(vc_condition_train[0:25])
print(vc_condition_test[0:25])

In [ ]:
vc_condition_train[vc_condition_train < 10].shape, vc_condition_test[vc_condition_test < 10].shape

In [ ]:
vc_condition_test[0:25].index.isin(vc_condition_train[0:25].index).all()


الشروط العليا هي نفسها تقريبًا لكلتا مجموعتي البيانات. أكثر من نصف الحالات تحدث أقل من 10 مرات.



لأن الحالة مرتبطة باسم الدواء 


In [ ]:
conditions_drugs_train = df_train.groupby(['condition'])['drugName'].nunique().sort_values(ascending=False)
conditions_drugs_test = df_test.groupby(['condition'])['drugName'].nunique().sort_values(ascending=False)

print(conditions_drugs_train[0:25])
print(conditions_drugs_test[0:25])


أعلى 15 شرطًا لكل دواء في كلتا مجموعتي البيانات هي نفسها. تجدر الإشارة إلى خيار Not Listed / Othe، لذا فإن NaNs كانت مفقودة بالفعل. وتجدر الإشارة أيضًا إلى أخطاء الزاحف مثل "3</span> وجد المستخدمون هذا التعليق مفيدًا". سنقوم بإصلاحه لاحقًا في مرحلة المعالجة المسبقة. 



أما بالنسبة لعمود اسم الدواء


In [ ]:
vc_drug_train = df_train['drugName'].value_counts()
vc_drug_test = df_test['drugName'].value_counts()

print(vc_drug_train[0:25])
print(vc_drug_test[0:25])

In [ ]:
vc_drug_test[0:15].index.isin(vc_drug_train[0:15].index).all()

In [ ]:
vc_drug_train[vc_drug_train < 10].shape, vc_drug_test[vc_drug_test < 10].shape


حسنا، تقريبا نفس الظروف. Top15 المخدرات هي نفسها. أكثر من نصف الأدوية تحدث أقل من 10 مرات.


In [ ]:
print(df_train['condition'].\
      iloc[df_train['condition'].astype(str).\
           apply(lambda str: len(str.split())).sort_values(ascending = False).index[0:10]])
print(df_train['condition'][24040:24041].apply(lambda str: len(str.split())))


الجزء التالي هو المراجعات. دعونا نلقي نظرة على عدد قليل


In [ ]:
df_train['review'][17]

In [ ]:
df_test['review'][42]

In [ ]:
df_train['review'][100000] # what a guy

In [ ]:
df_test['review'][53766 - 1] 

In [ ]:
df_train['review'][161297 - 1] 


أول ما يلفت الأنظار هو "#039" للفاصلة العليا. بعد ذلك، بعض أوامر التنسيق مثل '\r'، '\n'. سنقوم أيضًا بحذف هذه.



الميزة التالية هي ميزة العد المفيدة


In [ ]:
df_train['usefulCount'].describe()

In [ ]:
df_test['usefulCount'].describe()

In [ ]:
_, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 6))
sns.distplot(df_train['usefulCount'], ax=axes[0], norm_hist=True);
axes[0].set(xlabel='usefulCount_train', ylabel='count');
sns.distplot(df_test['usefulCount'], ax=axes[1], norm_hist=True);
axes[1].set(xlabel='usefulCount_test', ylabel='count');

In [ ]:
plt.figure(figsize=(12,9))

sns.distplot(df_train['usefulCount'], color='blue', kde=False, norm_hist=True)
sns.distplot(df_test['usefulCount'], color='green', kde=False, norm_hist=True)

plt.xlabel('usefulCount')
plt.ylabel('Counts')

plt.show()


تبدو التوزيعات متشابهة جدًا بالنسبة لمجموعات بيانات التدريب والاختبار. 



أما بالنسبة للتوزيع نفسه، فإن المشكلة في 'usefulCount' هي أن التوزيع منحرف بذيول طويلة. المعيار هو 36 عندما يكون المتوسط ​​27-28. "الأعداد المفيدة" مرتبطة بالحالة والمخدرات. بالنسبة للحالة الشائعة، هناك عدد أكبر بكثير من الأشخاص الذين قرأوا المراجعات ومن الواضح أن "الأعداد المفيدة" أعلى بكثير. سنحاول التعامل معها لاحقا.



التاريخ:


In [ ]:
print(np.min(df_train['date']),np.max(df_train['date']))
print(np.min(df_test['date']),np.max(df_test['date']))


حسنًا. الوقت للهدف.


In [ ]:
print(df_train['target'].value_counts())
print(df_train['target'].value_counts(normalize=True))

In [ ]:
print(df_test['target'].value_counts())
print(df_test['target'].value_counts(normalize=True))


الطبقات غير متوازنة.



من الآمن الآن أن نقول إن عينات الاختبار والتدريب مشتقة من توزيع واحد. دعونا نسلسلهم ونواصل التحليل.


In [ ]:
df_all = pd.concat([df_train,df_test]).reset_index(drop=True)
mask = df_all.index < df_train.shape[0]
df_all['istrain'] = False
df_all['istrain'][mask] = True


الفرز حسب الوقت


In [ ]:
df_all.sort_values(by='date',
        ascending=True).head(10)

لنقم بإنشاء ميزات الوقت الرئيسية


In [ ]:
df_all['year'] = df_all['date'].dt.year
df_all['month'] = df_all['date'].dt.month
df_all['dom'] = df_all['date'].dt.day
df_all['dow'] = df_all['date'].dt.weekday
df_all.drop('date', axis=1, inplace=True)

In [ ]:
# Graphics in SVG format are more sharp and legible
%config InlineBackend.figure_format = 'svg'

In [ ]:
_, axes = plt.subplots(nrows=1, ncols=2, figsize=(11, 4))
sns.countplot(x='year', hue='target', ax=axes[0], data=df_all[df_all['istrain']]);
axes[0].set(xlabel='year_train', ylabel='count');
sns.countplot(x='year', hue='target', ax=axes[1], data=df_all[df_all['istrain'] == False]);
axes[1].set(xlabel='year_test', ylabel='count');

In [ ]:
sns.countplot(x='year', hue='target', data=df_all);


مثيرة للاهتمام. يمكن رؤية 3 مجموعات من السنوات: 
1. 2008
2. 2009-2014
3. 2015-2017
ومما له أهمية خاصة أن عدد المراجعات السلبية قد زاد بشكل ملحوظ منذ عام 2015. وكان عدد المراجعات السلبية والمحايدة قبل عام 2015 متطابقًا تقريبًا.


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x='month', hue='target', data=df_all);


سيكون الشهر عديم الفائدة


In [ ]:
plt.figure(figsize=(9,4))
sns.countplot(x='dom', hue='target', data=df_all);


لا شيء مثير للاهتمام (يوجد 7 أشهر فقط في السنة و31 يومًا).


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x='dow', hue='target', data=df_all);


نفس الشيء. لن تساعد ميزات الوقت بمفردها (باستثناء السنة). لكننا سوف نتحقق من ذلك.



العودة إلى الأعداد المفيدة


In [ ]:
df_all.groupby(['target'])['usefulCount'].describe()


متوقع. غالبًا ما تحصل المراجعات الإيجابية على "عدد مفيد" أكثر من المراجعات السلبية. 


In [ ]:
del vc_drug_test, vc_drug_train, \
    vc_condition_train, vc_condition_test, conditions_drugs_test, conditions_drugs_train


لتلخيص ، 
1. هناك ~ 1300 قيمة مفقودة في عمود الشرط في مجموعتي البيانات. لقد اكتشفنا أيضًا خطأ في الزاحف في هذا العمود.
2. تم استخلاص مجموعات البيانات من توزيع واحد، وهو ما تم إثباته من خلال ميزات الوقت والأعداد المفيدة والهدف.
3. يجب أن نضع في اعتبارنا أن الفئات المستهدفة تكون غير متوازنة عندما نختار المقياس ونخصص النموذج.
4. ميزات الوقت تقريبًا غير مجدية (باستثناء السنة). قمنا بدراسة 3 مجموعات من السنوات: 1. 2008، 2. 2009-2014، 3. 2015-2017. ومما له أهمية خاصة أن عدد المراجعات السلبية قد زاد بشكل ملحوظ منذ عام 2015. وكان عدد المراجعات السلبية والمحايدة قبل عام 2015 متطابقًا تقريبًا.
5. هناك العديد من الأشياء التي يتعين علينا تصحيحها في المراجعات مثل مشكلة الفاصلة العليا وأوامر التنسيق.
6. تبدو ميزة الكونت المفيد مفيدة. ولكن هناك مشكلة مع عدد الأشخاص الذين يحتاجون إلى بعض "أسماء الأدوية" في حالة أو أخرى.



# 3. المعالجة المسبقة للبيانات واختيار المقاييس والنماذج


لنبدأ بالشرط نظرًا لأن هذا العمود يحتوي على قيم مفقودة.


In [ ]:
df_all[pd.isnull(df_all['condition'])].head(10)

In [ ]:
print(pd.isnull(df_all['condition']).sum())


كما تتذكر، هناك شرط "غير مدرج / غير ذلك". 


In [ ]:
print(df_all[df_all['condition'] == 'Not Listed / Othe'].shape)
df_all[df_all['condition'] == 'Not Listed / Othe'].head(10)


هناك أيضًا بعض أخطاء الزاحف في عمود الشرط:


In [ ]:
df_all['condition'].value_counts().tail(20)


"وجد المستخدمون هذا التعليق مفيدًا." - لنحسب مقدار هذه الأخطاء في التعبيرات العادية.


In [ ]:
import re

df_all['condition'].str.contains(re.compile('users found this comment helpful')).sum()


حسنًا، مقدار الأخطاء\القيم المفقودة في عمود الشرط صغير جدًا. ووفقا للمراجعات، يمكن ملاحظة أن هؤلاء الأشخاص لديهم ظروف مختلفة. من الأفضل حذفهم جميعاً.


In [ ]:
mask = (pd.isnull(df_all['condition'])) | (df_all['condition'] == 'Not Listed / Othe') |\
                 (df_all['condition'].str.contains(re.compile('users found this comment helpful')))

In [ ]:
mask.sum()

In [ ]:
df_all.shape

In [ ]:
df_all = df_all.drop(df_all[mask].index).reset_index(drop=True)

In [ ]:
df_all.shape


بالنسبة لميزة الشرط، سنطبق CountVectorizer، والذي يمكنه افتراضيًا تنفيذ جميع الإجراءات الضرورية مثل السلاسل الصغيرة، باستثناء علامات الترقيم و')(/'، وما إلى ذلك. كل ما يتعين علينا معرفته هو عدد الكلمات في "الشرط" الأطول لاختيار نطاق ngram.


In [ ]:
print('The number of words in the longest condition:', df_all['condition'].astype(str).\
      apply(lambda str: len(str.split())).sort_values(ascending = False).iloc[0])
print(df_all['condition'].iloc[df_all['condition'].astype(str).\
                               apply(lambda str: len(str.split())).sort_values(ascending = False).index[0]])


اسم الدواء. دعونا مرة أخرى نلقي نظرة على القيم.


In [ ]:
df_all['drugName'].head(15)

In [ ]:
df_all['drugName'].tail(15)


أما بالنسبة للحالة، فلنجد أطول اسم دواء.


In [ ]:
print(df_all['drugName'].iloc[df_all['drugName'].astype(str).\
                               apply(lambda str: len(str.split())).sort_values(ascending = False).index[1]])
print(df_all['drugName'].iloc[df_all['drugName'].astype(str).\
                               apply(lambda str: len(str.split())).sort_values(ascending = False).index[4]])


حسنًا، الحد الأقصى لنطاق ngram سيكون 10.



الوقت للمراجعات.



لنقم بإنشاء WordClouds.


In [ ]:
# !pip isntall WordCloud
# !conda install -c conda-forge wordcloud 
from wordcloud import WordCloud, STOPWORDS

def plot_wordcloud(data, title):
    wordcloud = WordCloud(background_color='black', stopwords = STOPWORDS, max_words = 100, max_font_size = 100, 
                    random_state = 42, width=1280, height=720)
    wordcloud.generate(str(data))
    
    plt.figure(figsize=(9, 6))

    plt.imshow(wordcloud, interpolation="bilinear");

    plt.title(title, fontdict={'size': 20, 'color': 'black', 
                                  'verticalalignment': 'bottom'})
    
    plt.axis('off')
    plt.tight_layout()  


في الحوسبة، الكلمات المتوقفة هي الكلمات التي يتم تصفيتها قبل أو بعد معالجة بيانات اللغة الطبيعية (النص). على الرغم من أن "الكلمات المتوقفة" تشير عادةً إلى الكلمات الأكثر شيوعًا في اللغة، إلا أنه لا توجد قائمة عالمية واحدة للكلمات المتوقفة التي تستخدمها جميع أدوات معالجة اللغة الطبيعية، وفي الواقع لا تستخدم جميع الأدوات مثل هذه القائمة. تتجنب بعض الأدوات على وجه التحديد إزالة كلمات التوقف هذه لدعم البحث عن العبارة. 
الاقتباس: https://en.wikipedia.org/wiki/Stop_words


In [ ]:
plot_wordcloud(df_all['review'], 'Wordcloud for both datasets')


دعونا نجمع المراجعات حسب الهدف


In [ ]:
plot_wordcloud(df_all['review'][df_all['target'] == 0], 'Wordcloud for negative reviews')
plot_wordcloud(df_all['review'][df_all['target'] == 1], 'Wordcloud for neutral reviews')
plot_wordcloud(df_all['review'][df_all['target'] == 2], 'Wordcloud for positive reviews')


حسنًا، يبدو أن الكلمات المفردة لن تساعدنا كثيرًا. فقط عدة كلمات مثل فظيع، فظيع. 
هل يمكنك رؤية الفرق؟


كما نتذكر، هناك مشكلة مع الفواصل العليا. سنقوم باستبدالها برمز الفضاء


In [ ]:
df_all['review'] = df_all['review'].str.replace('&#039;', " ", regex=False);

In [ ]:
df_all['review'][17]


بخير. سنستخدم TfidfVectorizer للمراجعات التي تحتوي على كلمات توقف. سنجرب 3 نطاقات ngram: (1,1)، (1،3)، (2،3). 



أما بالنسبة لميزة الوقت، فسنقوم أولاً بتضمينها جميعًا مع التحويل المناسب (أوه لسنوات؛ سين_كوس تراسنفورم لأيام_الأسبوع، أيام_الشهر، الشهر)


In [ ]:
def add_time_features(df):
    df['dow_sin'] = df['dow'].apply(lambda ts: np.sin(2*np.pi*ts/7.))
    df['dow_cos'] = df['dow'].apply(lambda ts: np.cos(2*np.pi*ts/7.))
    
    df['dom_sin'] = df['dom'].apply(lambda ts: np.sin(2*np.pi*ts/31.))
    df['dom_cos'] = df['dom'].apply(lambda ts: np.cos(2*np.pi*ts/31.))
    
    df['month_sin'] = df['month'].apply(lambda ts: np.sin(2*np.pi*ts/12.))
    df['month_cos'] = df['month'].apply(lambda ts: np.cos(2*np.pi*ts/12.))

    df.drop(['month', 'dom', 'dow'], axis=1, inplace=True)

    return df

In [ ]:
df_all = add_time_features(df_all)
df_all.head()


في البداية، سنقوم فقط بقياس "العدد المفيد" باستخدام StandardScaler 



اختيار النموذج.
مع الأخذ في الاعتبار 
1. حجم المهمة (كمية الميزات بعد المعالجة المسبقة - مضيعة للوقت باستخدام RandomForestClassifier وGBM وما إلى ذلك)
2. حجم مجموعة المراجعات (حوالي 160 ألف مراجعة في مجموعة بيانات القطار)
3. وجود الميزات العددية (لا يمكن ببساطة استخدام شيء مثل NaiveBayesClassifier)
4. إمكانيات اللاب توب (حزين)
5. تعليمات بعدم الغوص عميقًا (NN، أساليب محددة لمهام محددة، وهو التحليل العاطفي)
قررت العمل مع الانحدار اللوجستي متعدد الحدود. لقد عملنا معها عدة مرات. أعتقد أنه ليست هناك حاجة لوصف الإيجابيات والسلبيات مرة أخرى.



اختيار متري.
حسنًا، نظرًا لوجود تصنيف متعدد الحدود + فئات غير متوازنة:
0. سلبي - 0.251032
1. محايد - 0.147305
2. إيجابي - 0.601663
لذلك، الدقة البسيطة ليست فكرة جيدة. في هذه الحالة، من الأفضل اختيار الدقة\الاستدعاء أو حتى درجة F1 "المرجحة". إليك الوصف من وثائق sklearn https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html#sklearn.metrics.accuracy_score 
"المرجح":
احسب المقاييس لكل تصنيف، وابحث عن متوسطها المرجح حسب الدعم (عدد المثيلات الحقيقية لكل تصنيف). 



حسنًا. دعونا نستمر


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV

class LemmaTokenizer(object):
    def __init__(self):
        self.wnl = WordNetLemmatizer()
    def __call__(self, doc):
        return [self.wnl.lemmatize(t) for t in word_tokenize(doc)]

In [ ]:
X_train = df_all[df_all['istrain']]
X_test = df_all[df_all['istrain'] == False]
y_train = df_all['target'][df_all['istrain']]
y_test = df_all['target'][df_all['istrain'] == False]

X_train.drop(['istrain','target'], axis=1, inplace=True)
X_test.drop(['istrain','target'], axis=1, inplace=True)


خط أنابيب. الوقت ينفد، الموعد النهائي قريب، لذلك هذا هو أفضل إصدار عمل يمكنني القيام به الآن :)


In [ ]:
class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.key]

class NumberSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[[self.key]]
        
uc_transformer =  Pipeline(steps = [
                ('selector_uc', NumberSelector(key = 'usefulCount')),
                ('scaler_uc', StandardScaler())
            ])

dow_sin_transformer =  Pipeline(steps = [
                ('selector_dow_sin', NumberSelector(key = 'dow_sin')),
                ('scaler_dow_sin', StandardScaler())
            ])
dow_cos_transformer =  Pipeline(steps = [
                ('selector_dow_cos', NumberSelector(key = 'dow_cos')),
                ('scaler_dow_cos', StandardScaler())
            ])

dom_sin_transformer =  Pipeline(steps = [
                ('selector_dom_sin', NumberSelector(key = 'dom_sin')),
                ('scaler_dom_csin', StandardScaler())
            ])
dom_cos_transformer =  Pipeline(steps = [
                ('selector_dom_cos', NumberSelector(key = 'dom_cos')),
                ('scaler_dom_cos', StandardScaler())
            ])

month_sin_transformer =  Pipeline(steps = [
                ('selector_month_sin', NumberSelector(key = 'month_sin')),
                ('scaler_month_sin', StandardScaler())
            ])
month_cos_transformer =  Pipeline(steps = [
                ('selector_month_cos', NumberSelector(key = 'month_cos')),
                ('scaler_month_cos', StandardScaler())
            ])

cond_tranformer = Pipeline(steps = [
                ('selector_cond', TextSelector(key='condition')),
                ('cv_cond', CountVectorizer(stop_words='english', ngram_range=(1,8)))
            ])

drug_tranformer = Pipeline(steps = [
                ('selector_drug', TextSelector(key='drugName')),
                ('cv_cond', CountVectorizer(stop_words='english', ngram_range=(1,10)))
            ])

y_transformer = Pipeline(steps = [
                ('selector_y', NumberSelector(key='year')),
                ('ohe_y', OneHotEncoder(handle_unknown='ignore'))
            ])

rev_tranformer = Pipeline(steps = [
                ('selector_rev', TextSelector(key='review')),
                ('tfidf_rev',TfidfVectorizer(stop_words='english', 
                                             ngram_range=(1,3),
                                             max_features = 100000
                                            )
                )
            ])

preprocessor = FeatureUnion([
        ('usefulCount', uc_transformer),
        ('dow_sin', dow_sin_transformer), 
        ('dow_cos', dow_cos_transformer), 
        ('dom_sin', dom_sin_transformer), 
        ('dom_cos', dom_sin_transformer), 
        ('month_sin', month_sin_transformer), 
        ('month_cos', month_cos_transformer), 
        ('condition', cond_tranformer),
        ('drugName', drug_tranformer),
        ('year', y_transformer),
        ('review', rev_tranformer)
    ])


mlog = Pipeline(steps = [('preprocessor', preprocessor),
                         ('logreg', LogisticRegression(random_state=42, solver = 'lbfgs', multi_class='multinomial'))])



دعونا نحاول التنبؤ

In [ ]:
%%time

warnings.filterwarnings('ignore')

mlog.fit(X_train,  y_train)


درجة f1 المرجحة لمجموعة بيانات الاختبار


In [ ]:
y_pred = mlog.predict(X_test)
print('Weighted f1 score for test dataset',f1_score(y_test, y_pred, average='weighted'))


دعونا نحسب الدقة أيضًا


In [ ]:
print('Accuracy score', accuracy_score(y_test, y_pred))


ومصفوفة الارتباك: https://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html#sphx-glr-auto-examples-model-selection-plot-confusion-matrix-py


In [ ]:
import itertools

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

In [ ]:
plot_confusion_matrix(confusion_matrix(y_test, y_pred),classes = [0, 1, 2])


حسنًا، أعتقد أنه بالنسبة للنموذج الافتراضي فلا بأس. الوقت للسيرة الذاتية



# 4. إنشاء ميزات جديدة ووصف لهذه العملية



حسنًا، سنحاول الآن حذف الميزات التي قد لا يكون لها أي تأثير. كما ذكرنا، يجب إزالة جميع ميزات الوقت باستثناء السنة. علاوة على ذلك، بدلاً من عمود السنة، سنقوم بإنشاء 3 ميزات منطقية لثلاث مجموعات موصوفة أعلاه.
نحتاج أيضًا إلى معالجة المشكلة باستخدام الأعداد المفيدة. الفكرة البسيطة هي أن تقسم على كمية الشروط


In [ ]:
def add_new_features(df):
    df['year1'] = df['year'] == 2008
    df['year2'] = (df['year'] < 2015) & (df['year'] > 2008)
    df['year3'] = (2014 < df['year']) & (df['year'] < 2018)

    df.drop('year', axis=1, inplace=True)

    return df

df_all.drop(['dow_sin', 'dow_cos', 'dom_sin', 'dom_cos', 'month_sin', 'month_cos'], axis = 1, inplace = True)
df_all = add_new_features(df_all)

In [ ]:
X_train = df_all[df_all['istrain']]
X_test = df_all[df_all['istrain'] == False]
y_train = df_all['target'][df_all['istrain']]
y_test = df_all['target'][df_all['istrain'] == False]

X_train.drop(['istrain','target'], axis=1, inplace=True)
X_test.drop(['istrain','target'], axis=1, inplace=True)

In [ ]:
y1_transformer = Pipeline(steps = [
                ('selector_y1', NumberSelector(key='year1'))
            ])

y2_transformer = Pipeline(steps = [
                ('selector_y2', NumberSelector(key='year2'))
            ])

y3_transformer = Pipeline(steps = [
                ('selector_y3', NumberSelector(key='year3'))
            ])

preprocessor2 = FeatureUnion([
        ('usefulCount', uc_transformer),
        ('condition', cond_tranformer),
        ('drugName', drug_tranformer),
        ('year1', y1_transformer),
        ('year2', y2_transformer),
        ('year3', y3_transformer),
        ('review', rev_tranformer)
    ])


mlog2 = Pipeline(steps = [('preprocessor', preprocessor2),
                         ('logreg', LogisticRegression(random_state=42, solver = 'lbfgs', multi_class='multinomial', ))])



دعونا التحقق من ذلك


In [ ]:
%%time
mlog2.fit(X_train,  y_train)

In [ ]:
y_pred2 = mlog2.predict(X_test)
print('Weighted f1 score for test dataset',f1_score(y_test, y_pred2, average='weighted'))

In [ ]:
print('Accuracy score', accuracy_score(y_test, y_pred2))


حسنًا، على الرغم من الطبقات غير المتوازنة، فإن الدقة تبدو جيدة.


In [ ]:
plot_confusion_matrix(confusion_matrix(y_test, y_pred2),classes = [0, 1, 2])


حسنا، يبدو أفضل!



# 5. التحقق من صحة وتعديل المعلمات الفائقة للنموذج



نظرًا لأن فصولنا غير متوازنة واختبار القطار جاء من نفس التوزيع، فسنقوم بإجراء StratifiedKFold بثلاثة تقسيمات


In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
cv = StratifiedKFold(n_splits = 3)


حان الوقت لضبط المعلمات الفائقة. 
Logreg، إذا كنت تتذكر، لديه طريقتان للتنظيم: التنظيم l1، l2. ولكن لا يوجد سوى 3 حلول تدعم المسائل متعددة الفئات وجميعها تعمل فقط مع l2. أعتقد أنه ليست هناك حاجة لوصف ما هو تنظيم l2.
أما بالنسبة لـ Tfidf، فكما ذكرنا أعلاه، سنجرب عدة نطاقات ngram_ranges. دع الحد الأقصى من الميزات يظل ثابتًا (100 ألف)


In [ ]:
param_grid = {
    'preprocessor__review__tfidf_rev__ngram_range': [(1,1), (1,3), (2,3)],
    'logreg__C': [0.1, 0.5, 1.0, 2],
}

grid_search = GridSearchCV(mlog2, param_grid, cv=cv, scoring = 'f1_weighted',verbose = 20, n_jobs = -1)

In [ ]:
grid_search.fit(X_train,y_train)

In [ ]:
print('grid search params:', grid_search.best_params_)
print('grid search score:', grid_search.best_score_)


# 6. رسم منحنيات التدريب والتحقق من الصحة



لا يوجد وقت لهذا الكتاب المقدس



# 7. التنبؤ بالعينات الاختبارية أو المحتجزة 



انتهى الموعد النهائي ولكنني أجمع هنا: mlog2.set_params(grid_search.best_params_) لا يعمل...


In [ ]:
mlog2.set_params(logreg__C = 2, preprocessor__review__tfidf_rev__ngram_range = (1, 3))
mlog2.fit(X_train, y_train)
y_pred_final = mlog2.predict(X_test)

In [ ]:
print('Weighted f1 score for test dataset', f1_score(y_test, y_pred_final, average='weighted'))

In [ ]:
print('Accuracy score', accuracy_score(y_test, y_pred_final))

انخفضت النتيجة. حسنا، لم يبق شيء للقيام به. يجب أن تؤدي السيرة الذاتية الأفضل (المزيد من الانقسامات، والمزيد من قيم المعلمات) إلى تحسين النتائج، لأنها مشتقة من نفس التوزيع.



# 8. الاستنتاجات



في هذا المشروع، ندرس مشكلة التصنيف العاطفي استنادًا إلى مجموعة بيانات UCI ML Drug Review. لقد حاولنا تطبيق أكبر قدر ممكن مما تعلمناه خلال الأشهر الثلاثة الماضية. ومع ذلك، تظهر النتائج. ليست جيدة جدًا، لكن... هيا، إنها مجرد بداية. خط أنابيب بسيط مع هندسة مميزة أعطانا النتيجة المذكورة أعلاه. وكما هو مذكور في معايير التقييم، يجب أن أصف قيمة المشروع. حسنًا ، إنه ضخم بالنسبة لي. ملء النماذج لا يتناسب مع مشروعك الخاص. أتمنى أن تشاركوني الرأي
حالات التطبيق الممكنة؟ لا احد. ولكن كخط أساسي بالنسبة لي أن أتعلم شيئًا عن التحليل العاطفي، فلا بأس بذلك.
وبطبيعة الحال، هناك الكثير من الطرق للتحسين: تقنيات البرمجة اللغوية العصبية، والتعلم العميق، وما إلى ذلك. ولكن قبل كل شيء، لا بد لي من إنهاء المهمة.
شكرا للقراءة!